In [ ]:
# Standard libraries
import os
import sys
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "code").exists()), Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "code"))

from bayesian_optimization import load_param_spaces_from_dir, run_bayes_search
from data_loader import CropYieldDataLoader
from leave_one_year_out import yield_prediction
from scorer import rmse_scorer
from utils import plot_metric_by_algorithm, scatterplot_visualization


## **Data Loader**

In [ ]:
# Example usage:
target_region = ['Cherkaska', 'Chernihivska', 'Chernivetska', 'Dnipropetrovska', 'Donetska', 'Kharkivska', 'Khmelnytska', 'Kirovohradska', 
                 'Luhanska', 'Lvivska', 'Mykolaivska', 'Odeska', 'Poltavska', 'Rivnenska', 'Sumska', 'Ternopilska', 'Vinnytska', 
                 'Volynska', 'Zakarpatska', 'Zaporizka', 'Zhytomyrska']

n_regions = len(target_region)
region_names = target_region

years = np.arange(2010, 2023 + 1)
selected_years = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

combined_file_path = str(PROJECT_ROOT / 'data/processed/combined_data.csv')
crop_yield_file_path = str(PROJECT_ROOT / 'data/processed/crop_yield.csv')

data_loader = CropYieldDataLoader(target_region, combined_file_path, crop_yield_file_path)
data_loader.load_data()

X_selected, y_selected = data_loader.filter_years(selected_years)


result_save_path = str(PROJECT_ROOT / 'results/11_20')
param_dir = str(PROJECT_ROOT / 'configs/search_spaces')  # Folder containing all YAML files
params_path = str(PROJECT_ROOT / 'results/11_20/hyperparams')  # Folder to save results

## **Bayesian Optimization**

In [ ]:
param_spaces = load_param_spaces_from_dir(param_dir)

# Run BayesSearchCV for all models using RMSE
run_bayes_search(X_selected, y_selected, param_spaces, params_path, rmse_scorer, n_regions=n_regions)

## **ML**

1. X.shape: (308, 104)
- 21 regions * 14 years (2010 ~ 2023)
- 13 indicators * 8 months (3 ~ 10)
2. y.shape: (308,)
- wheat yield

### **SVM**

In [ ]:
estimator = SVR
model_name = 'svm'
metric_name = 'RMSE'

yield_prediction(estimator, params_path, model_name, metric_name, X_selected, y_selected, target_region, selected_years, result_save_path)

### **RF**

In [ ]:
estimator = RandomForestRegressor
model_name = 'rf'
metric_name = 'RMSE'

yield_prediction(estimator, params_path, model_name, metric_name, X_selected, y_selected, target_region, selected_years, result_save_path)

### **Gradient Boosting**

In [ ]:
estimator = GradientBoostingRegressor
model_name = 'gb'
metric_name = 'RMSE'

yield_prediction(estimator, params_path, model_name, metric_name, X_selected, y_selected, target_region, selected_years, result_save_path)

### **DT**

In [ ]:
estimator = DecisionTreeRegressor
model_name = 'dt'
metric_name = 'RMSE'

yield_prediction(estimator, params_path, model_name, metric_name, X_selected, y_selected, target_region, selected_years, result_save_path)

## **KNN Regression**

In [ ]:
estimator = KNeighborsRegressor
model_name = 'knn'
metric_name = 'RMSE'

yield_prediction(estimator, params_path, model_name, metric_name, X_selected, y_selected, target_region, selected_years, result_save_path)

## **XGBoost**

In [ ]:
estimator = XGBRegressor
model_name = 'xgb'
metric_name = 'RMSE'

yield_prediction(estimator, params_path, model_name, metric_name, X_selected, y_selected, target_region, selected_years, result_save_path)

## **Scatter Plot and Correlation**

In [ ]:
# Load the CSV file
file_path = str(PROJECT_ROOT / 'results/12_20/rf_results.csv')
df = pd.read_csv(file_path)

# Extract the filename without extension for the plot title
file_name = os.path.splitext(os.path.basename(file_path))[0]

# Visualize results, print correlation, and R² value
scatterplot_visualization(df, file_name)

## **Visualization**

In [ ]:
# Example usage
directory = str(PROJECT_ROOT / 'results/12_20')  # Replace with your directory path
metric = 'RMSE'  # Replace with the desired metric (e.g., 'RMSE', 'MAE')
plot_metric_by_algorithm(directory, metric)